# Waymo Dataset Exploration

### SDSC Expanse Environment Setup
#### Total Cores Requested: 32
#### Total Memory Requested: 150 GB
#### SparkSession Configuration Justification:
##### Driver Memory: 2 GB (The Driver only orchestrates the tasks and holds aggregated results, so it needs minimal memory).
##### Executor Instances: 31. Calculated using the formula: Total Cores (32) - 1 (Driver) = 31 Executors.
##### Executor Memory: 4 GB. Calculated using the formula: (Total Memory (150 GB) - Driver Memory (2 GB)) / 31 Executors = 4.77 GB. I conservatively allocated 4g per executor to leave room for the Singularity container's OS overhead and prevent Out-Of-Memory (OOM) crashes.

In [40]:
import sys
from pyspark.sql import SparkSession

# Define the library path
lib_path = "/expanse/lustre/scratch/apasvenskas/temp_project/python_libs"

# Ensure the driver has it
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

# Build the new session with executorEnv.PYTHONPATH
spark = SparkSession.builder \
    .appName("Waymo_Trajectory_Forecasting") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.instances", 31) \
    .config("spark.executorEnv.PYTHONPATH", lib_path) \
    .getOrCreate()

print("Fresh Spark session built with native worker paths!")

Fresh Spark session built with native worker paths!


In [41]:
import glob
from pyspark.sql import Row

## Collect all the files

In [42]:
data_dir = "/expanse/lustre/scratch/apasvenskas/temp_project/waymo_training_30gb/"
file_paths = glob.glob(data_dir + "*.tfrecord*")
print(f"Preparing to process {len(file_paths)} files using 31 executors...")

Preparing to process 60 files using 31 executors...


#### Get and covert the data from Waymo data set ( with a custom PySpark worker script designed to read, decode, and filter the Waymo Open Motion Dataset)

In [43]:
def distributed_python_parser(file_partition):
    import sys
    import struct
    
    # Inject path for the Waymo Protobuf definitions
    lib_path = "/expanse/lustre/scratch/apasvenskas/temp_project/python_libs"
    if lib_path not in sys.path:
        sys.path.insert(0, lib_path)
        
    from waymo_open_dataset.protos import scenario_pb2
    
    parsed_rows = []
    for path in file_partition:
        try:
            # Bypass TensorFlow entirely 
            with open(path, 'rb') as f:
                while True:
                    # Read 8-byte length header
                    length_bytes = f.read(8)
                    if len(length_bytes) != 8:
                        break 
                    
                    length = struct.unpack('<Q', length_bytes)[0]
                    f.read(4) # Skip 4-byte CRC header
                    raw_data = f.read(length) # Read the Protobuf data
                    f.read(4) # Skip 4-byte CRC footer
                    
                    # DECODE AND FILTER
                    scenario = scenario_pb2.Scenario()
                    scenario.ParseFromString(raw_data)
                    
                    # Intersections only (Traffic Lights present)
                    if len(scenario.dynamic_map_states) == 0:
                        continue 

                    # Extract vehicles with full 91-frame history
                    for track in scenario.tracks:
                        if track.object_type == 1 and len(track.states) == 91:
                            history_states = track.states[:11]
                            future_states = track.states[11:]

                            if all(state.valid for state in history_states):
                                parsed_rows.append(Row(
                                    scenario_id=scenario.scenario_id,
                                    track_id=track.id,
                                    past_x=[s.center_x for s in history_states],
                                    past_y=[s.center_y for s in history_states],
                                    future_x=[s.center_x for s in future_states],
                                    future_y=[s.center_y for s in future_states]
                                ))
        except Exception as e:
            print(f"Skipped corrupted file {path} due to error: {e}")
            continue
            
    return iter(parsed_rows)

#### Parallelize the file list

In [44]:
print("Might take a few minutes...")
waymo_rdd = spark.sparkContext.parallelize(file_paths, numSlices=31).mapPartitions(distributed_python_parser)
ml_df = waymo_rdd.toDF()

Might take a few minutes...


#### Show the data

In [45]:
from pyspark.sql.functions import col, count, when, isnan, explode, size

In [46]:
ml_df.printSchema()
ml_df.show(5)

root
 |-- scenario_id: string (nullable = true)
 |-- track_id: long (nullable = true)
 |-- past_x: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- past_y: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- future_x: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- future_y: array (nullable = true)
 |    |-- element: double (containsNull = true)

+----------------+--------+--------------------+--------------------+--------------------+--------------------+
|     scenario_id|track_id|              past_x|              past_y|            future_x|            future_y|
+----------------+--------+--------------------+--------------------+--------------------+--------------------+
|7ae2d6e15faa7089|    1024|[2095.03784179687...|[-1544.9141845703...|[2103.07958984375...|[-1531.9429931640...|
|7ae2d6e15faa7089|    1025|[2079.42333984375...|[-1572.5478515625...|[2084.52734375, 2...|[-1563.0043945312...|
|7ae2d6e1

In [47]:
total_vehicles = ml_df.count()
print(f"Total intersection vehicles: {total_vehicles}")

Total intersection vehicles: 832346


#### Descriptive Statistics

In [48]:
ml_df.select("scenario_id", "track_id").describe().show()

+-------+----------------+------------------+
|summary|     scenario_id|          track_id|
+-------+----------------+------------------+
|  count|          832346|            832346|
|   mean|        Infinity|1198.5660458511245|
| stddev|             NaN| 1111.459155151483|
|    min|1001f7ed426203e1|                 0|
|    max|fffffb2b1c5d65c8|             14519|
+-------+----------------+------------------+



#### Checking for missing values

In [49]:
null_counts = []
for c, t in ml_df.dtypes:
    if t in ('double', 'float'):
        null_counts.append(count(when(isnan(c) | col(c).isNull(), c)).alias(c))
    else:
        null_counts.append(count(when(col(c).isNull(), c)).alias(c))

ml_df.select(null_counts).show()

+-----------+--------+------+------+--------+--------+
|scenario_id|track_id|past_x|past_y|future_x|future_y|
+-----------+--------+------+------+--------+--------+
|          0|       0|     0|     0|       0|       0|
+-----------+--------+------+------+--------+--------+



#### Check for duplicates

In [50]:
distinct_count = ml_df.dropDuplicates(["scenario_id", "track_id"]).count()
total_obs = ml_df.count()
print(f"Total rows: {total_obs}")
print(f"Distinct rows: {distinct_count}")

if total_obs == distinct_count:
    print("No duplicates")
else:
    print(f"There are {total_obs - distinct_count} duplicates.")

Total rows: 832346
Distinct rows: 832346
No duplicates


***************

## Waymo Data Visualization

In [51]:
# !pip install --target=/expanse/lustre/scratch/apasvenskas/temp_project/python_libs plotly

In [52]:
import sys
# Ensure scratch libraries are used
lib_path = "/expanse/lustre/scratch/apasvenskas/temp_project/python_libs"
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

import plotly.graph_objects as go
# from IPython.display import HTML, display
from plotly.subplots import make_subplots
from pyspark.sql.functions import col

print("Aggregating data with Spark...")

Aggregating data with Spark...


#### Bar plot

In [53]:
busy_scenarios_rows = ml_df.groupBy("scenario_id").count().orderBy(col("count").desc()).limit(10).collect()
bar_x = [row['scenario_id'][:6] + '...' for row in busy_scenarios_rows]
bar_y = [row['count'] for row in busy_scenarios_rows]

In [54]:
fig = go.Figure(data=[
    go.Bar(x=bar_x, y=bar_y, marker_color='skyblue')
])

In [55]:
fig.update_layout(
    title="Top 10 Busiest Intersections",
    xaxis_title="Scenario ID (Truncated)",
    yaxis_title="Number of Tracked Vehicles",
    template="plotly_white",
    height=500,
    width=800
)

In [56]:
fig.show(renderer="iframe")

#### Histogram

In [57]:
hist_data = [row['count'] for row in ml_df.groupBy("scenario_id").count().select("count").collect()]

In [58]:
fig = go.Figure(data=[go.Histogram(x=hist_data, nbinsx=20, marker_color='coral')])
fig.update_layout(title="Intersection Density", xaxis_title="Vehicles per Scene")

fig.show(renderer="iframe")
fig

#### Scatter plot

In [59]:
v = ml_df.select("past_x", "past_y", "future_x", "future_y").limit(1).collect()[0]

In [60]:
fig = go.Figure(data=[
    go.Scatter(x=v['past_x'], y=v['past_y'], mode='markers+lines', marker=dict(color='blue', size=8), name='Past (1.1s)'),
    go.Scatter(x=v['future_x'], y=v['future_y'], mode='markers+lines', marker=dict(color='red', size=5), opacity=0.6, name='Future (8.0s)'),
    go.Scatter(x=[v['past_x'][0]], y=[v['past_y'][0]], mode='markers', marker=dict(color='green', symbol='star', size=18), name='Start Point')
])

In [61]:
fig.update_layout(title="Detailed Vehicle Trajectory", xaxis_title="X (m)", yaxis_title="Y (m)", template="plotly_white")

fig.show(renderer="iframe")
fig

## Getting the executors

In [62]:
print(spark.sparkContext.uiWebUrl)

http://exp-4-19.expanse.sdsc.edu:4040


In [63]:
import requests

In [64]:
app_info = requests.get("http://localhost:4040/api/v1/applications").json()
app_id = app_info[0]['id']

In [65]:
executors = requests.get(f"http://localhost:4040/api/v1/applications/{app_id}/executors").json()

In [66]:
active_count = len([e for e in executors if e.get('isActive')])
print(f"Total Active Executors: {active_count}")

Total Active Executors: 1


In [67]:
print(f"{'ID':<5} | {'Host/Port':<35} | {'Active':<8} | {'Cores':<6} | {'Max Mem (GB)':<15}")
print("-" * 80)

ID    | Host/Port                           | Active   | Cores  | Max Mem (GB)   
--------------------------------------------------------------------------------


#### Because the 32 cores were allocated on a single SDSC Expanse node via Slurm, Spark dynamically defaulted to Local Mode. Rather than incurring network overhead by splitting tasks across 31 separate executor JVMs, Spark pooled all 32 cores directly into the Driver for highly efficient multi-threaded parallel processing.

In [68]:
for e in executors:
    # Convert memory from bytes to GB safely
    max_mem_gb = round(e.get('maxMemory', 0) / (1024**3), 2)
    
    # Extract the fields safely using .get()
    e_id = e.get('id', 'N/A')
    e_host = e.get('hostPort', 'N/A')
    e_active = str(e.get('isActive', False))
    e_cores = e.get('totalCores', 0)
    
    print(f"{e_id:<5} | {e_host:<35} | {e_active:<8} | {e_cores:<6} | {max_mem_gb:<15}")

driver | exp-4-19.expanse.sdsc.edu:38707     | True     | 32     | 1.02           
